##

> Week 1: From text to input embeddings

###

> 1. Data

In [47]:
import tiktoken
import kagglehub
from dotenv import load_dotenv
from pathlib import Path
import torch

from src.tokenizer import create_dataloader

torch.manual_seed(1234)

In [48]:
load_dotenv()
DATA_PATH = Path("../data")
DATA_PATH.mkdir(parents=True, exist_ok=True)
DATA_NAME = "kaushaltiwari/tiny-shakespeare"

# Download Data

path = kagglehub.dataset_download(
    handle=DATA_NAME,
    output_dir=DATA_PATH.as_posix(),
)

In [49]:
# Tokenizer using gt2
full_path = DATA_PATH / "tiny-shakespeare.txt"
with open(full_path, "r", encoding="utf-8") as f:
    text = f.read()

###

> 2. Tokens

A toy word-level tokenizer: a dictionary that maps each word to an id, with `<unk>` for unknown words.

In [50]:
# To make it simple I will create a Toy Dictioary 
toy_text= "I love my kids with all my hear"
splitted_text = toy_text.split(" ")
tokenizer = {word: idx for idx, word in enumerate(set(splitted_text))}
tokenizer["<unk>"] = len(tokenizer)
print(f"My tokenizer size `{len(tokenizer)}` {tokenizer}")

My tokenizer size `8` {'my': 0, 'all': 1, 'kids': 2, 'I': 3, 'love': 4, 'hear': 5, 'with': 6, '<unk>': 7}


###

> 3. Manual implementation

Building (mimicking) a vector embedding by hand.

####

> What is an embedding?

An embedding maps each token id to a dense vector of real numbers.
It is stored as a matrix W of shape (N, M), where row i is the vector of token i:

- N = vocabulary size of the tokenizer (50257 for GPT-2 BPE), regardless of
how many of those tokens actually appear in our text.

- M = embedding dimension (e.g. 768 in GPT-2 small). It is a hyperparameter
we choose before training.

W starts with random values and is learned together with the rest of the model.
If we trained our own tokenizer on Tiny Shakespeare, N would be that tokenizer's
vocabulary size. A word-level vocabulary is impractical: any unseen word would
be out of vocabulary, which is why subword tokenizers like BPE are used.


In [51]:
# Embdding Matrix 
dim = 3
ids = torch.tensor([tokenizer[w] for w in splitted_text])
W = torch.arange(len(tokenizer) * dim, dtype=torch.float32).reshape(len(tokenizer), 3)
W

tensor([[ 0.,  1.,  2.],
        [ 3.,  4.,  5.],
        [ 6.,  7.,  8.],
        [ 9., 10., 11.],
        [12., 13., 14.],
        [15., 16., 17.],
        [18., 19., 20.],
        [21., 22., 23.]])

In [52]:
# Create One hot encoding 
onehot = []
for idx in ids.tolist():
    row = [0 * len(tokenizer)] * len(tokenizer)
    row[idx] = 1
    onehot.append(row)


In [53]:
# Now create the embedding layer using the embedding matrix W
torch.tensor(onehot).float() @ W

tensor([[ 9., 10., 11.],
        [12., 13., 14.],
        [ 0.,  1.,  2.],
        [ 6.,  7.,  8.],
        [18., 19., 20.],
        [ 3.,  4.,  5.],
        [ 0.,  1.,  2.],
        [15., 16., 17.]])

###

> 4. Linear implementation

The same lookup, now as a `Linear` layer applied to one-hot vectors.

In [54]:
# Transform the ids into one-hot encoding using PyTorch's built-in function
onehot = torch.nn.functional.one_hot(ids, num_classes=len(tokenizer)).float()

# Now create the embedding layer using the embedding matrix W
linear = torch.nn.Linear(len(tokenizer), dim, bias=False)
linear.weight = torch.nn.Parameter(W.T)
linear(onehot)

tensor([[ 9., 10., 11.],
        [12., 13., 14.],
        [ 0.,  1.,  2.],
        [ 6.,  7.,  8.],
        [18., 19., 20.],
        [ 3.,  4.,  5.],
        [ 0.,  1.,  2.],
        [15., 16., 17.]], grad_fn=<MmBackward0>)

###

> 5. Implementation with torch

Now using `torch.nn.Embedding`.

###

> 6. Data loader

This creates the dataset for training and testing. The challenge here is choosing `batch_size`, `max_length` and `stride`.

In [55]:
print(f"This is the dataset we are using: {DATA_NAME}")

# MaxLength is the maximum length of the sequence we want to feed into the model.
max_length = 128

# Generate Data
dataloader = create_dataloader(
    data=text,
    batch_size=64,
    max_length=max_length,
    stride=max_length, # This avoid Overlapping Sequences
)

data_iter = iter(dataloader)
inputs, target = next(data_iter)
print("Inputs:\n", inputs)
print("target:\n", target)

This is the dataset we are using: kaushaltiwari/tiny-shakespeare
Inputs:
 tensor([[ 8001, 14210,  1654,  ...,    32,  1014,     0],
        [  656, 46548,    25,  ...,  2694,   338,   287],
        [   25,   198,    46,  ...,  2323,    11,   355],
        ...,
        [  338,  7432,    11,  ...,   475,   788,  3446],
        [   25,   198,  5779,  ...,   867,    11,   393],
        [ 1359,    13,   198,  ...,   198,  5211,   407]])
target:
 tensor([[14210,  1654,   286,  ...,  1014,     0,  1309],
        [46548,    25,   339,  ...,   338,   287,   502],
        [  198,    46,    11,  ...,    11,   355,   314],
        ...,
        [ 7432,    11,   198,  ...,   788,  3446,   466],
        [  198,  5779,    11,  ...,    11,   393,  2073],
        [   13,   198,   198,  ...,  5211,   407, 42848]])


###

> 7. Embedding layer and positional embedding

####

> Token embedding

In [56]:
# Embeddings
output_dim = 256
vocab_size = tiktoken.get_encoding("gpt2").n_vocab  # 50257

# Embedding matrix W of shape (vocab_size, output_dim), randomly initialized.
# It is learned during training together with the rest of the model.
token_embedding_layer = torch.nn.Embedding(num_embeddings=vocab_size, embedding_dim=output_dim)

# Lookup: each token id in `inputs` selects its row of W
token_embedding = token_embedding_layer(inputs)
batch_size, seq_len, emb_dim = token_embedding.shape
print(f"Embedding output [batch_size, seq_len, emb_dim]: {token_embedding.shape}")
print(f"{batch_size} sequences, {seq_len} tokens each, {emb_dim} numbers per token")


Embedding output [batch_size, seq_len, emb_dim]: torch.Size([64, 128, 256])
64 sequences, 128 tokens each, 256 numbers per token


####

> Positional embedding

In [57]:
# Generate Positional Econding 
post_embedding_layer = torch.nn.Embedding(num_embeddings=max_length, embedding_dim=output_dim)
post_embedding_layer

Embedding(128, 256)

####

> Input embedding (token + position)

From this point adding the positional encoding into the vector embedding to shift | move the vector for the transformer to identified the best vector position
we just sum them into the vector space , this would affect all the batches 

In [58]:
input_embedding = token_embedding + post_embedding_layer(torch.arange(seq_len))
input_embedding.shape

torch.Size([64, 128, 256])

###

> 7. Neural Network 

####

> Basic Neuron

In [59]:
import torch.nn.functional as F

y = torch.tensor([1])       # True Label
x1 = torch.tensor([1.1])    # Input Feature
w = torch.tensor([0.5])     # Weight
b = torch.tensor([0.0])     # Bias

z = w * x1 + b              # net input
a = torch.sigmoid(z)        # activation output

loss = F.binary_cross_entropy(a, y.float())
print(f"Loss: {loss.item()}")


Loss: 0.45549246668815613


####

> Applying propagation

In [60]:
from torch.autograd import grad 

y = torch.tensor([1])
x1 = torch.tensor([1.1])
w1 = torch.tensor([0.5], requires_grad=True)   
b = torch.tensor([0.0], requires_grad=True)

z = w1 * x1 + b
a = torch.sigmoid(z)
loss = F.binary_cross_entropy(a, y.float())

print(f"Loss: {loss.item()}")

# Compute gradients
grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

print(f"Gradient of loss w.r.t weight: {grad_L_w1[0].item()}")
print(f"Gradient of loss w.r.t bias: {grad_L_b[0].item()}")

Loss: 0.45549246668815613
Gradient of loss w.r.t weight: -0.402450829744339
Gradient of loss w.r.t bias: -0.3658643960952759


In [61]:
loss.backward()

print(w1.grad)
print(b.grad)

tensor([-0.4025])
tensor([-0.3659])


####

> Implementing Neural Network with torch.nn

In [62]:
from src.nets import NeuralNetwork
torch.manual_seed(1234)

# Crate a model 
model = NeuralNetwork(50,3)
print(model)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)


In [63]:
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total number of trainable parameters: {num_params}")

Total number of trainable parameters: 2213


In [64]:
# Showing Weights and Biases of the first layer
first_layer = model.layers[0]
print("Weights of the first layer:\n", first_layer.weight)
print("Biases of the first layer:\n", first_layer.bias)

Weights of the first layer:
 Parameter containing:
tensor([[-0.1332, -0.0277, -0.0679,  ..., -0.1229,  0.1086,  0.0872],
        [ 0.0715,  0.1128,  0.0520,  ...,  0.0622,  0.0734,  0.0088],
        [ 0.0410,  0.0629, -0.0165,  ...,  0.1351,  0.0509, -0.0523],
        ...,
        [-0.0326, -0.1390,  0.0027,  ...,  0.0155, -0.0064,  0.0576],
        [-0.0843, -0.0869,  0.0910,  ...,  0.1006, -0.0516,  0.0199],
        [ 0.0013, -0.0454,  0.0163,  ..., -0.0539, -0.0676,  0.1381]],
       requires_grad=True)
Biases of the first layer:
 Parameter containing:
tensor([ 0.0461,  0.0244, -0.0678, -0.0378,  0.1271,  0.0593,  0.0638, -0.0139,
        -0.1346,  0.0286,  0.1091,  0.1008,  0.0430, -0.1087,  0.0680, -0.0928,
         0.1095,  0.0845,  0.1046, -0.1082,  0.0285, -0.0725,  0.1102,  0.0860,
         0.1388,  0.0071, -0.0483,  0.1147,  0.0980,  0.0036],
       requires_grad=True)


In [65]:
torch.manual_seed(123)

model = NeuralNetwork(50, 3)
print(model.layers[0].weight)

Parameter containing:
tensor([[-0.0577,  0.0047, -0.0702,  ...,  0.0222,  0.1260,  0.0865],
        [ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        ...,
        [-0.0787,  0.1259,  0.0803,  ...,  0.1218,  0.1303, -0.1351],
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,  0.0344, -0.0971, -0.0509]],
       requires_grad=True)


In [66]:
# Testing 
X = torch.randn(5, 50)  # 5 samples, each with 50 features
output = model(X)
print("Output of the model:\n", output)

Output of the model:
 tensor([[-0.1860,  0.1045, -0.2019],
        [-0.0790,  0.0769, -0.0375],
        [-0.3337,  0.1545, -0.3261],
        [-0.0796,  0.0874, -0.0568],
        [-0.1403,  0.2365, -0.1998]], grad_fn=<AddmmBackward0>)


In [67]:
with torch.no_grad():
    output = model(X)
    print("Output of the model (no gradient tracking):\n", output)

Output of the model (no gradient tracking):
 tensor([[-0.1860,  0.1045, -0.2019],
        [-0.0790,  0.0769, -0.0375],
        [-0.3337,  0.1545, -0.3261],
        [-0.0796,  0.0874, -0.0568],
        [-0.1403,  0.2365, -0.1998]])


In [68]:
with torch.no_grad():
    out = torch.softmax(model(X), dim=1)
print(out)

tensor([[0.3011, 0.4026, 0.2963],
        [0.3114, 0.3640, 0.3246],
        [0.2749, 0.4480, 0.2771],
        [0.3120, 0.3687, 0.3192],
        [0.2941, 0.4287, 0.2771]])


In [69]:
# Creating extra data 
X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])

X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6],
])

y_train = torch.tensor([0, 0, 0, 1, 1])
y_test = torch.tensor([0, 1])

In [70]:
class ToyDataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ids = ToyDataset(X_train, y_train)
test_ids = ToyDataset(X_test, y_test)
len(train_ids)

5

In [81]:
# Create Data Loader
train_loader = torch.utils.data.DataLoader(train_ids, batch_size=2, shuffle=True, drop_last=True)
test_loader = torch.utils.data.DataLoader(test_ids, batch_size=2, shuffle=False, drop_last=True) 

In [82]:
for batch_idx, (X_batch, y_batch) in enumerate(train_loader):
    print(f"Batch {batch_idx + 1}:")
    print("X_batch:\n", X_batch)
    print("y_batch:\n", y_batch)
    print()

Batch 1:
X_batch:
 tensor([[-1.2000,  3.1000],
        [ 2.3000, -1.1000]])
y_batch:
 tensor([0, 1])

Batch 2:
X_batch:
 tensor([[ 2.7000, -1.5000],
        [-0.9000,  2.9000]])
y_batch:
 tensor([1, 0])



In [83]:
# Doing a Typical Training Loop

model = NeuralNetwork(X_train.shape[1], len(torch.unique(y_train)))
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
num_epochs = 3

for epoch in range(num_epochs):

    model.train()  # Set the model to training mode
    for batch_idx, (X_batch, y_batch) in enumerate(train_loader):
        logtis = model(X_batch)    # Trained 
        loss = F.cross_entropy(logtis, y_batch)  # Compute loss
        optimizer.zero_grad()  # Zero the gradients
        loss.backward()        # Backpropagation
        optimizer.step()       # Update weights

        ### LOGGING
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx+1:03d}/{len(train_loader):03d}"
              f" | Train/Val Loss: {loss:.2f}")

    model.eval()
    # Optional model evaluation


Epoch: 001/003 | Batch 001/002 | Train/Val Loss: 0.67
Epoch: 001/003 | Batch 002/002 | Train/Val Loss: 0.76
Epoch: 002/003 | Batch 001/002 | Train/Val Loss: 0.72
Epoch: 002/003 | Batch 002/002 | Train/Val Loss: 0.69
Epoch: 003/003 | Batch 001/002 | Train/Val Loss: 0.68
Epoch: 003/003 | Batch 002/002 | Train/Val Loss: 0.67


In [87]:
# Check on test without gradient tracking
model.eval()
with torch.no_grad():
    outputs = model(X_train)
print(f"Model outputs on test data:\n{outputs}")

Model outputs on test data:
tensor([[ 0.1175, -0.0226],
        [ 0.0952, -0.0079],
        [ 0.0622,  0.0081],
        [-0.0932, -0.0429],
        [-0.0854, -0.0496]])


In [88]:
torch.set_printoptions(sci_mode=False)
probas = torch.softmax(outputs, dim=1)
print(probas)

predictions = torch.argmax(probas, dim=1)
print(predictions)

tensor([[0.5350, 0.4650],
        [0.5258, 0.4742],
        [0.5135, 0.4865],
        [0.4874, 0.5126],
        [0.4910, 0.5090]])
tensor([0, 0, 0, 1, 1])


In [89]:
predictions = torch.argmax(outputs, dim=1)
predictions

tensor([0, 0, 0, 1, 1])

In [90]:
predictions == y_train

tensor([True, True, True, True, True])

In [93]:
torch.sum(predictions == y_train)

tensor(5)

In [ ]:
def compute_accuracy(model, dataloader) -> float:

    model.eval()  # Set the model to evaluation mode
    correct = 0
    total_examples = 0

    for _, (features, labels) in enumerate(dataloader):

        # Deactivate gradient tracking for evaluation
        with torch.no_grad():
            logits = model(features)

            # Update counts
            predictions = torch.argmax(logits, dim=1)
            compare = labels == predictions
            correct += torch.sum(compare)
            total_examples += len(compare)

    return (correct / total_examples).item()

# Compute Accuracy on the training | Testing set
print(f"accuracy in training {compute_accuracy(model, train_loader)}")
print(f"accuracy in testing {compute_accuracy(model, test_loader)}")

# Saving Model Weights 


accuracy in training 1.0
accuracy in testing 1.0


In [100]:
# Save and load model weights
torch.save(model.state_dict(), "/llm-studies/models/model_weights_week1.pth")

# Load again the model 
model2 = NeuralNetwork(X_train.shape[1], len(torch.unique(y_train)))
model2.load_state_dict(torch.load("/llm-studies/models/model_weights_week1.pth", weights_only=True))

<All keys matched successfully>